[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S06/2_Practica/en_clase/S06_04_metricas_validacion_en_clase.ipynb)

# S06 · 04 — scikit-learn básico: métricas y validación · versión en clase

⏱ 45 min en clase · Sprint 6 · M3 (UD 3.1 y 3.2)

**Contexto TurisData.** Un modelo que "acierta el 90 %" puede ser inútil o, peor, engañoso. Antes de comparar modelos hay que saber **cómo medirlos honestamente**: con qué métrica, con qué partición y con qué *baseline* de referencia. Trabajamos con `reservas_turisdata.csv` (6000 reservas; objetivo `cancelada`).

**Este cuaderno tiene 3 ejercicios** (1 a 3), los que se hacen en el aula. La regresión con MAE, RMSE y R² (ejercicio 2), elegir la profundidad del árbol (ejercicio 3), la partición temporal, `GridSearchCV`, el mini-reto de la tabla de resultados y las preguntas de reflexión están en el cuaderno completo. Sus ejercicios 1, 4 y 5 son los 3 de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, cross_val_score, StratifiedKFold,
                                     TimeSeriesSplit, GridSearchCV)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score)

SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "mes_llegada"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]


def hacer_preprocesado(num=columnas_num, cat=columnas_cat):
    """Receta estándar: imputar + escalar (num) e imputar + one-hot (cat)."""
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
    ])


X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
print("Tasa de cancelación:", y.mean().round(3))
print("Listo ✔")

### 👀 A2 · EJEMPLO — El baseline: predecir siempre lo mismo

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué exactitud tiene el baseline?
2. ¿Qué predice siempre el baseline y por qué acierta más de la mitad?
3. Si un modelo tuviera una exactitud de 0,55, ¿valdría la pena?

In [ ]:
tonto = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Exactitud del baseline:", round(tonto.score(X_test, y_test), 3))

### ✏️ A3 · EJERCICIO 1 de 3 — Regresión logística frente al baseline

**Tu turno:** Crea `logistica`, un `Pipeline` con `hacer_preprocesado()` y `LogisticRegression(max_iter=1000)`. Ajústalo con train y guarda en `acc_logistica` la exactitud en test y en `mejora` la diferencia con `tonto.score(...)`.

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Cuánto valen `acc_logistica` y `mejora`?
2. ¿Qué mide `mejora` y por qué es la cifra importante?
3. ¿Por qué ajustamos el pipeline con `X_train` y puntuamos con `X_test`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: Pipeline([("prep", ...), ("clf", ...)]).fit(...); logistica.score(X_test, y_test)
logistica = ...
acc_logistica = ...
mejora = ...

assert acc_logistica > 0.65, "La regresión logística debería superar 0,65 de exactitud"
assert mejora > 0.05, "Debería mejorar al baseline en más de 5 puntos"
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — Exactitud, precisión, recall y F1

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. Con la precisión (0,653): de cada 100 reservas que el modelo anuncia como «se cancelará», ¿cuántas se cancelan de verdad?
2. Con el recall (0,608): de cada 100 reservas que se cancelan, ¿cuántas detecta?
3. ¿Qué error costaría más a TurisData: no detectar una cancelación o anunciar una que no ocurre?

In [ ]:
pred = logistica.predict(X_test)
print(f"Exactitud {accuracy_score(y_test, pred):.3f} · Precisión {precision_score(y_test, pred):.3f} "
      f"· Recall {recall_score(y_test, pred):.3f} · F1 {f1_score(y_test, pred):.3f}")

### 👀 A5 · EJEMPLO — Curva de sobreajuste: la profundidad del árbol

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Qué le pasa a la curva de entrenamiento al aumentar la profundidad?
2. ¿Y a la de prueba? ¿Hasta qué profundidad mejora?
3. ¿Qué indica que las dos curvas se separen cada vez más?

In [ ]:
prep_arbol = hacer_preprocesado()
Xtr_p = prep_arbol.fit_transform(X_train)
Xte_p = prep_arbol.transform(X_test)
profundidades = range(1, 21)
acc_tr, acc_te = [], []
for p in profundidades:
    arbol = DecisionTreeClassifier(max_depth=p, random_state=SEMILLA).fit(Xtr_p, y_train)
    acc_tr.append(arbol.score(Xtr_p, y_train))
    acc_te.append(arbol.score(Xte_p, y_test))
plt.figure(figsize=(8, 4))
plt.plot(profundidades, acc_tr, marker="o", label="Entrenamiento")
plt.plot(profundidades, acc_te, marker="s", label="Prueba")
plt.title("Sobreajuste: profundidad del árbol vs. exactitud")
plt.xlabel("Profundidad máxima del árbol")
plt.ylabel("Exactitud")
plt.legend()
plt.show()

### 👀 A6 · EJEMPLO — Validación cruzada de 5 folds

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Entre qué valores oscilan las 5 exactitudes?
2. ¿Qué dice la media 0,688 y qué dice la desviación 0,015?
3. ¿Por qué la media de la validación cruzada (0,688) no coincide con la exactitud en test (0,672)?

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
puntuaciones = cross_val_score(logistica, X_train, y_train, cv=cv, scoring="accuracy")
print("Exactitud por fold:", puntuaciones.round(3))
print(f"Media {puntuaciones.mean():.3f} ± {puntuaciones.std():.3f}")

### ✏️ A7 · EJERCICIO 2 de 3 — AUC con validación cruzada

**Tu turno:** Calcula `auc_cv`: el array de puntuaciones de validación cruzada de `logistica` con `scoring="roc_auc"` y el mismo `cv` (5 folds). Guarda su media en `auc_medio`.

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Cuánto vale `auc_medio` y cuántas puntuaciones hay en `auc_cv`?
2. ¿Qué significa un AUC de 0,5 y qué uno de 0,755?
3. ¿Por qué se calcula sobre `X_train` y no sobre `X_test`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: cross_val_score(logistica, X_train, y_train, cv=cv, scoring="roc_auc")
auc_cv = ...
auc_medio = ...

assert len(auc_cv) == 5, "Deben ser 5 puntuaciones (5 folds)"
assert 0.65 < auc_medio < 0.95, "El AUC medio debería estar entre 0,65 y 0,95"
print("✔ Ejercicio 2 correcto")

### 👀 A8 · EJEMPLO — Una columna que delata el resultado

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Qué AUC sale sin la columna y con ella?
2. ¿Por qué es sospechoso que el AUC suba tanto?
3. ¿Qué pasaría en producción con ese modelo?

In [ ]:
rng = np.random.default_rng(SEMILLA)
X_fuga = X.copy()
X_fuga["reembolso_emitido"] = np.where(y == 1, rng.choice([1, 1, 1, 0], len(y)), 0)  # casi = cancelada
prep_fuga = hacer_preprocesado(num=columnas_num + ["reembolso_emitido"])
modelo_fuga = Pipeline([("prep", prep_fuga), ("clf", LogisticRegression(max_iter=1000))])
auc_fuga = cross_val_score(modelo_fuga, X_fuga, y, cv=cv, scoring="roc_auc").mean()
auc_real = cross_val_score(logistica, X, y, cv=cv, scoring="roc_auc").mean()
print(f"AUC sin la columna: {auc_real:.3f} · AUC con la columna que 'delata' el resultado: {auc_fuga:.3f}")

### ✏️ A9 · EJERCICIO 3 de 3 — Detectar las columnas con fuga

**Tu turno:** De la lista `candidatas`, indica en `columnas_con_fuga` (lista) cuáles **no se conocerían en el momento de hacer la reserva** y por tanto provocarían fuga.

**Debe salir:** ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Qué columnas provocan fuga y qué tienen en común?
2. ¿Por qué `tarifa`, `antelacion_dias` y `cancelaciones_previas` no son fuga?
3. ¿Qué pregunta te sirve para decidir con cualquier columna?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
candidatas = {
    "antelacion_dias": "días entre reservar y llegar (se conoce al reservar)",
    "importe_devuelto": "euros reembolsados al cliente tras cancelar",
    "tarifa": "flexible o no reembolsable (se elige al reservar)",
    "fecha_cancelacion": "fecha en que el cliente canceló (vacía si no canceló)",
    "cancelaciones_previas": "cancelaciones de reservas anteriores del cliente",
}
# pista: pregúntate: ¿existe esta información antes de saber si cancela?
columnas_con_fuga = ...

assert set(columnas_con_fuga) == {"importe_devuelto", "fecha_cancelacion"}, "Piensa en qué datos solo existen DESPUÉS de cancelar"
print("✔ Ejercicio 3 correcto")

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S06/2_Practica/para_casa/S06_04_metricas_validacion.ipynb)** trae los ejercicios de regresión (MAE, RMSE y R²), de elegir la profundidad del árbol y de partición temporal, la búsqueda de hiperparámetros con `GridSearchCV`, el mini-reto de la tabla de resultados y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y, de `scikit-learn`, los modelos, las métricas, los validadores cruzados y el preprocesado.
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - Lee `reservas_turisdata.csv` (6000 reservas), añade `mes_llegada`, define las listas de columnas y la función `hacer_preprocesado()` (la receta de la práctica anterior: imputar y escalar los números, imputar y one-hot las categorías).
    - Separa `X` e `y` y reparte entrenamiento y prueba (80 % / 20 %, estratificado y con semilla), igual que en la práctica anterior.
- **Qué debe salir:**
    - `Tasa de cancelación: 0.459` y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. La API de scikit-learn y el *baseline*

#### A2 · Ejemplo — El baseline: predecir siempre lo mismo

**Idea.** Todos los modelos de scikit-learn siguen el mismo patrón: `modelo.fit(X_train, y_train)` aprende, `modelo.predict(X_nuevo)` predice y `modelo.score(X, y)` mide con la métrica por defecto. Antes de nada, un **baseline**: un modelo tonto (p. ej., «predecir siempre la clase más frecuente»). Si tu modelo no lo supera claramente, no aporta nada.

- **Qué hace:**
    - `DummyClassifier(strategy="most_frequent")` es el modelo tonto: predice siempre la clase más frecuente de train. `.fit(X_train, y_train)` lo «entrena» (solo cuenta cuál es).
    - `tonto.score(X_test, y_test)` calcula su **exactitud** (proporción de aciertos) en test y `round(..., 3)` la redondea a 3 decimales.
- **Qué debe salir:**
    - `Exactitud del baseline: 0.541`.
- **Si falla:**
    - `NameError: name 'X_train' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué exactitud tiene el baseline?* → **0,541**.
2. *¿Qué predice siempre el baseline y por qué acierta más de la mitad?* → Predice «no se cancela» (la clase más frecuente: solo el 45,9 % de las reservas se cancela), así que acierta en las que no se cancelan: el **54,1 %**.
3. *Si un modelo tuviera una exactitud de 0,55, ¿valdría la pena?* → Casi nada: apenas mejora el baseline (0,541). Un modelo solo aporta si lo supera **claramente**.

</details>

#### A3 · Ejercicio 1 — Regresión logística frente al baseline

- **Qué hace:**
    - `Pipeline([("prep", hacer_preprocesado()), ("clf", LogisticRegression(max_iter=1000))])` encadena la receta de preprocesado y el modelo.
    - `logistica.fit(X_train, y_train)` lo ajusta con train; `logistica.score(X_test, y_test)` mide su exactitud en test (`acc_logistica`).
    - `mejora` es la diferencia entre esa exactitud y la del baseline: lo que el modelo aporta de verdad.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`. `acc_logistica` vale 0,672 y `mejora` 0,131 (13 puntos por encima del baseline).
- **Si falla:**
    - `La regresión logística debería superar 0,65 de exactitud` → falta el preprocesado en el pipeline o no has ajustado con `X_train`.
    - `Debería mejorar al baseline en más de 5 puntos` → `mejora` no es `acc_logistica - tonto.score(X_test, y_test)`.
    - `NameError: name 'tonto' is not defined` → falta ejecutar el ejemplo del baseline.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen `acc_logistica` y `mejora`?* → **0,672** y **0,131**.
2. *¿Qué mide `mejora` y por qué es la cifra importante?* → Cuánto aporta el modelo frente a no tener modelo: 0,672 − 0,541 = 0,131, **13 puntos**. La exactitud sola no dice si el modelo vale la pena; la comparación con el baseline, sí.
3. *¿Por qué ajustamos el pipeline con `X_train` y puntuamos con `X_test`?* → Porque test es el «examen sorpresa»: el modelo no debe haberlo visto al aprender. Y el pipeline aplica por dentro el preprocesado aprendido en train.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
logistica = Pipeline([("prep", hacer_preprocesado()), ("clf", LogisticRegression(max_iter=1000))])
logistica.fit(X_train, y_train)
acc_logistica = logistica.score(X_test, y_test)
mejora = acc_logistica - tonto.score(X_test, y_test)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. Métricas: ¿cómo de bueno es?

#### A4 · Ejemplo — Exactitud, precisión, recall y F1

**Clasificación** (predecir una clase). Con `positivo = cancelada`:
- **Exactitud (accuracy):** % de aciertos. Engaña si las clases están desbalanceadas.
- **Precisión:** de las reservas que decimos «se cancelará», ¿cuántas se cancelan de verdad?
- **Recall (sensibilidad):** de las que se cancelan, ¿cuántas detectamos?
- **F1:** media armónica de precisión y recall.

- **Qué hace:**
    - `logistica.predict(X_test)` predice la clase de cada reserva de test (0 o 1).
    - `accuracy_score`, `precision_score`, `recall_score` y `f1_score` comparan esas predicciones con la realidad (`y_test`); el `print` las escribe con 3 decimales.
- **Qué debe salir:**
    - `Exactitud 0.672 · Precisión 0.653 · Recall 0.608 · F1 0.630`.
- **Si falla:**
    - `NameError: name 'logistica' is not defined` → falta el ejercicio anterior (o su solución).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *Con la precisión (0,653): de cada 100 reservas que el modelo anuncia como «se cancelará», ¿cuántas se cancelan de verdad?* → Unas **65** (en test: de 513 reservas anunciadas, 335 se cancelaron).
2. *Con el recall (0,608): de cada 100 reservas que se cancelan, ¿cuántas detecta?* → Unas **61** (de 551 cancelaciones reales detecta 335; **se le escapan 216**).
3. *¿Qué error costaría más a TurisData: no detectar una cancelación o anunciar una que no ocurre?* → **Depende del negocio.** Si no detectar supone una habitación vacía, interesa un recall alto; si anunciar de más supone molestar al cliente o sobrevender, interesa una precisión alta. Por eso se miran las dos (y F1 las resume en una: 0,630, entre ambas).

</details>

## 3. Sobreajuste y sesgo-varianza

#### A5 · Ejemplo — Curva de sobreajuste: la profundidad del árbol

**Idea.** Un modelo muy flexible puede **memorizar** el entrenamiento (*sobreajuste*: alta varianza) y fallar con datos nuevos. Uno demasiado simple no captura el patrón (*infraajuste*: alto sesgo). La señal clásica: la exactitud en **train sigue subiendo** mientras la de **test se estanca o baja**. Lo vemos con un árbol de profundidad creciente.

- **Qué hace:**
    - `hacer_preprocesado()` prepara los datos una sola vez (`fit_transform` en train, `transform` en test).
    - El bucle `for p in profundidades` entrena un `DecisionTreeClassifier` con cada profundidad máxima (de 1 a 20) y guarda su exactitud en train (`acc_tr`) y en test (`acc_te`).
    - Las líneas de `plt` dibujan las dos curvas: entrenamiento y prueba.
- **Qué debe salir:**
    - Un gráfico con dos curvas. La de entrenamiento sube sin parar, de 0,626 a 0,993; la de prueba sube hasta 0,657 (profundidad 4) y luego baja hasta ≈ 0,60.
- **Si falla:**
    - `NameError: name 'hacer_preprocesado' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué le pasa a la curva de entrenamiento al aumentar la profundidad?* → **Sube sin parar**, de 0,626 con profundidad 1 hasta 0,993 con 20: el árbol acaba memorizando.
2. *¿Y a la de prueba? ¿Hasta qué profundidad mejora?* → Mejora hasta la profundidad **4** (≈ 0,657) y después **baja** hasta ≈ 0,60.
3. *¿Qué indica que las dos curvas se separen cada vez más?* → **Sobreajuste**: el árbol se aprende el entrenamiento de memoria y falla con datos nuevos. Con profundidad 20 acierta el 99 % de train y solo el 60 % de test.

</details>

## 4. Validación cruzada

#### A6 · Ejemplo — Validación cruzada de 5 folds

> **Cuidado:** elegir la profundidad mirando el conjunto de **test** ya es una pequeña fuga (el test deja de ser un examen sorpresa). Lo correcto es elegir con **validación cruzada sobre train** y usar test una sola vez al final.

**Idea.** En lugar de una única partición (que depende de la suerte), se divide train en *k* trozos (*folds*): se entrena con *k−1* y se valida con el restante, rotando. Obtenemos *k* puntuaciones: su **media** estima el rendimiento y su **desviación** la estabilidad. `StratifiedKFold` mantiene la proporción de clases en cada trozo.

- **Qué hace:**
    - `StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)` crea el reparto en 5 trozos, mezclando antes con semilla fija y manteniendo la proporción de canceladas en cada uno.
    - `cross_val_score(logistica, X_train, y_train, cv=cv, scoring="accuracy")` entrena y valida el pipeline 5 veces y devuelve las 5 exactitudes.
    - Los `print` escriben cada puntuación y su media ± desviación.
- **Qué debe salir:**
    - `Exactitud por fold: [0.678 0.675 0.699 0.676 0.712]`.
    - `Media 0.688 ± 0.015`.
- **Si falla:**
    - `NameError: name 'logistica' is not defined` → falta el ejercicio 1 (o su solución en la Parte B).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Entre qué valores oscilan las 5 exactitudes?* → Entre **0,675 y 0,712**.
2. *¿Qué dice la media 0,688 y qué dice la desviación 0,015?* → La media estima el rendimiento esperado (**≈ 69 %**); la desviación, la **estabilidad**: el resultado cambia ±1,5 puntos según qué datos caigan en cada trozo.
3. *¿Por qué la media de la validación cruzada (0,688) no coincide con la exactitud en test (0,672)?* → Porque se calculan con **datos distintos** (trozos de train frente a test) y cada partición tiene su propia suerte. La diferencia (1,6 puntos) es del tamaño de la desviación: son resultados compatibles.

</details>

#### A7 · Ejercicio 2 — AUC con validación cruzada

El **AUC ROC** mide si el modelo **ordena bien**: da probabilidades más altas a las reservas que se cancelan que a las que no. 0,5 es como el azar (el baseline) y 1 es perfecto. No depende de en qué punto cortemos «cancela / no cancela».

- **Qué hace:**
    - Misma llamada que en el ejemplo anterior, pero con `scoring="roc_auc"`: devuelve **5 puntuaciones**, una por fold.
    - `auc_medio = auc_cv.mean()` es su media.
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`. `auc_cv` vale `[0.741 0.744 0.758 0.754 0.776]` y `auc_medio` ≈ 0,755.
- **Si falla:**
    - `Deben ser 5 puntuaciones (5 folds)` → `auc_cv` no es el resultado de `cross_val_score` con `cv=cv`.
    - `El AUC medio debería estar entre 0,65 y 0,95` → has dejado `scoring="accuracy"` (la exactitud ronda 0,69) o falta `.mean()`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale `auc_medio` y cuántas puntuaciones hay en `auc_cv`?* → **≈ 0,755** y **5** puntuaciones (una por fold, de 0,741 a 0,776).
2. *¿Qué significa un AUC de 0,5 y qué uno de 0,755?* → 0,5 es **azar** (el baseline). 0,755 significa que, si tomas una reserva cancelada y otra no, el modelo da más probabilidad a la cancelada en **≈ 75 %** de los casos.
3. *¿Por qué se calcula sobre `X_train` y no sobre `X_test`?* → Para decidir y comparar modelos sin gastar test: test se usa **una sola vez al final**, como examen sorpresa.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
auc_cv = cross_val_score(logistica, X_train, y_train, cv=cv, scoring="roc_auc")
auc_medio = auc_cv.mean()
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 5. Fuga de datos (*data leakage*)

#### A8 · Ejemplo — Una columna que delata el resultado

**Idea.** Hay fuga cuando el modelo usa información que **no tendría en el momento de predecir** en la vida real. Resultado: métricas fantásticas en el laboratorio y fracaso en producción. Ejemplos: columnas creadas *después* del desenlace (p. ej. «importe reembolsado»), estadísticos calculados con todo el dataset o duplicados entre train y test.

Simulamos una columna con fuga: `reembolso_emitido`, que el sistema marca **solo cuando la reserva se cancela y se devuelve el dinero**. Ninguna persona real la conocería al reservar.

- **Qué hace:**
    - `X_fuga` es una copia de `X` con una columna nueva, `reembolso_emitido`, que vale 1 en tres de cada cuatro reservas canceladas y 0 en todas las demás.
    - Se entrena el mismo tipo de pipeline con esa columna (`modelo_fuga`) y sin ella (`logistica`), y se mide el AUC de cada uno con validación cruzada.
    - El `print` compara los dos AUC.
- **Qué debe salir:**
    - `AUC sin la columna: 0.753 · AUC con la columna que 'delata' el resultado: 0.937`.
- **Si falla:**
    - `NameError: name 'cv' is not defined` → falta ejecutar el ejemplo de la validación cruzada (sección 4).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC sale sin la columna y con ella?* → **0,753** sin ella y **0,937** con ella.
2. *¿Por qué es sospechoso que el AUC suba tanto?* → Porque la columna **casi es la respuesta**: vale 1 en la mayoría de las canceladas y 0 en las demás. El modelo ya no predice: lee la solución.
3. *¿Qué pasaría en producción con ese modelo?* → Que al reservar `reembolso_emitido` **aún no existe** (siempre vale 0): el modelo dejaría de acertar y rendiría peor que el de 0,753. Un resultado «demasiado bueno» es una señal de alarma.

</details>

#### A9 · Ejercicio 3 — Detectar las columnas con fuga

La regla práctica ante cada columna: **¿esta información existe en el momento de hacer la predicción?** Si solo se conoce *después* de saber si la reserva se cancela, es fuga.

- **Qué hace:**
    - `candidatas` es un diccionario con cada columna y una descripción de qué es y cuándo se conoce.
    - `columnas_con_fuga` es una lista con los **nombres** de las columnas que no se conocerían al reservar.
- **Qué debe salir:**
    - `✔ Ejercicio 3 correcto`. `columnas_con_fuga` vale `["importe_devuelto", "fecha_cancelacion"]` (en cualquier orden).
- **Si falla:**
    - `Piensa en qué datos solo existen DESPUÉS de cancelar` → la lista no coincide: sobra alguna columna que sí se conoce al reservar (`antelacion_dias`, `tarifa`, `cancelaciones_previas`) o falta alguna de las dos que delatan el resultado.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué columnas provocan fuga y qué tienen en común?* → **`importe_devuelto`** y **`fecha_cancelacion`**: solo existen **después** de que el cliente cancele (la fecha incluso está vacía si no canceló).
2. *¿Por qué `tarifa`, `antelacion_dias` y `cancelaciones_previas` no son fuga?* → Porque se conocen **al reservar**: la tarifa se elige entonces, la antelación es la distancia entre reservar y llegar, y las cancelaciones previas son del historial del cliente, anterior a esta reserva.
3. *¿Qué pregunta te sirve para decidir con cualquier columna?* → «**¿Existiría este dato en el momento de predecir?**» Si la respuesta es no, hay que quitarla del modelo.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
candidatas = {
    "antelacion_dias": "días entre reservar y llegar (se conoce al reservar)",
    "importe_devuelto": "euros reembolsados al cliente tras cancelar",
    "tarifa": "flexible o no reembolsable (se elige al reservar)",
    "fecha_cancelacion": "fecha en que el cliente canceló (vacía si no canceló)",
    "cancelaciones_previas": "cancelaciones de reservas anteriores del cliente",
}
columnas_con_fuga = ["importe_devuelto", "fecha_cancelacion"]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>